# Qwen3-0.6B Direct Preference Optimization (DPO) & Odds Ratio Preference Optimization (ORPO) Training Demo (No Eval)

This notebook demonstrates the training workflow of fine-tuning a model using Direct Preference Optimization (DPO) or Odds Ratio Preference Optimization (ORPO) with MaxText, without the evaluation steps. It is suitable for automated integration testing suites.

In [ ]:
try:
  import google.colab
  IN_COLAB = True
except ImportError:
  IN_COLAB = False

In [ ]:
if IN_COLAB:
    !git clone https://github.com/AI-Hypercomputer/maxtext.git
    %cd maxtext
    !pip install uv
    import os
    os.environ["UV_TORCH_BACKEND"]="cpu"
    !uv pip install -e .[tpu-post-train] --resolution=lowest
    !install_tpu_post_train_extra_deps

In [ ]:
import jax
import os
import sys
import subprocess
from etils import epath

sys.path.append(os.path.abspath("src"))

from maxtext.configs import pyconfig
from maxtext.trainers.post_train.dpo import train_dpo
from maxtext.utils.globals import MAXTEXT_PKG_DIR

if not jax.distributed.is_initialized():
    jax.distributed.initialize()
print(f"JAX version: {jax.__version__}")
print(f"JAX devices: {jax.devices()}")

In [ ]:
MODEL_NAME = "qwen3-0.6b"
TOKENIZER_PATH = "Qwen/Qwen3-0.6B"
HF_PATH = "Qwen/Qwen3-0.6B"
NUM_DPO_TRAINING_STEPS = 5
PER_DEVICE_BATCH_SIZE = 1
DPO_ALGORITHM = "dpo"  # Alignment algorithm to use. Can be set to "dpo" or "orpo"

BASE_DIR = "/tmp/dpo_demo_no_eval"
MODEL_CHECKPOINT_PATH = f"{BASE_DIR}/sft_baseline_checkpoint"
DPO_OUTPUT_DIR = f"{BASE_DIR}/dpo_output"
RUN_NAME = "dpo_qwen3_demo_no_eval"

os.makedirs(BASE_DIR, exist_ok=True)

In [ ]:
if not epath.Path(MODEL_CHECKPOINT_PATH).exists():
    print("Installing torch for conversion...")
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install",
            "torch", "--index-url", "https://download.pytorch.org/whl/cpu"
        ],
        check=True
    )

    print("Converting checkpoint from HuggingFace...")
    env = os.environ.copy()
    env["JAX_PLATFORMS"] = "cpu"

    subprocess.run(
        [
            sys.executable,
            "-m", "maxtext.checkpoint_conversion.to_maxtext",
            "src/maxtext/configs/base.yml",
            f"model_name={MODEL_NAME}",
            f"base_output_directory={MODEL_CHECKPOINT_PATH}",
            f"--hf_model_path={HF_PATH}",
            "use_multimodal=false",
            "scan_layers=true",
            "skip_jax_distributed_system=True",
        ],
        check=True,
        env=env
    )

    CONVERTED_SFT_CKPT = os.path.join(MODEL_CHECKPOINT_PATH, "0/items")
    print(f"Checkpoint converted to {CONVERTED_SFT_CKPT}")
else:
    CONVERTED_SFT_CKPT = os.path.join(MODEL_CHECKPOINT_PATH, "0/items")
    print(f"Model checkpoint already exists at {CONVERTED_SFT_CKPT}")

In [ ]:
dpo_config = pyconfig.initialize_pydantic([
    "",
    f"{MAXTEXT_PKG_DIR}/configs/post_train/dpo.yml",
    f"load_parameters_path={CONVERTED_SFT_CKPT}",
    f"model_name={MODEL_NAME}",
    f"base_output_directory={DPO_OUTPUT_DIR}",
    f"run_name={RUN_NAME}",
    f"tokenizer_path={TOKENIZER_PATH}",
    "dataset_type=hf",
    "hf_path=argilla/distilabel-intel-orca-dpo-pairs",
    "train_split=train",
    "hf_eval_split=train",
    "train_data_columns=['input', 'chosen', 'rejected']",
    f"per_device_batch_size={PER_DEVICE_BATCH_SIZE}",
    "max_target_length=1024",
    f"steps={NUM_DPO_TRAINING_STEPS}",
    f"eval_interval={NUM_DPO_TRAINING_STEPS}",
    "learning_rate=5e-7",
    f"dpo.algo={DPO_ALGORITHM}",
    "weight_dtype=bfloat16",
    "dtype=bfloat16",
    "scan_layers=True",
    "enable_checkpointing=True",
    "async_checkpointing=False",
])

print("Setting up DPO trainer state...")
trainer, mesh = train_dpo.setup_trainer_state(dpo_config)

print("Starting DPO training...")
trainer = train_dpo.train_model(dpo_config, trainer, mesh)
print("DPO training completed successfully!")

DPO_CHECKPOINT = os.path.join(DPO_OUTPUT_DIR, RUN_NAME, "checkpoints", str(NUM_DPO_TRAINING_STEPS))
print(f"DPO checkpoint saved at: {DPO_CHECKPOINT}")

In [ ]:
# Clean up directories created during training to free up disk space
import shutil
if os.path.exists(BASE_DIR):
    print(f"Clearing BASE_DIR to free up disk space: {BASE_DIR}")
    shutil.rmtree(BASE_DIR)
